# Week 1 — From LLM to Tool-Using AI Assistant
## Gerçek API kullanan mini agent: Weather Decision Assistant


### 0. Giriş: Bugün ne yapacağız?

Bu notebook’ta klasik bir chatbot yerine dış dünya ile konuşabilen bir LLM sistemi kuracağız.
Model kendi başına hava durumunu bilemez. Bu yüzden ona araçlar vereceğiz.
Bu araçlar Python fonksiyonları olacak. Model hangi fonksiyonu ne zaman çağıracağını seçecek.


### 1. Teori: LLM neden tool’a ihtiyaç duyar?



```
LLM = reasoning / language engine
Tool = external capability
Agent = LLM + tools + loop + state
MCP = tools için standart bağlantı katmanı
```


Function calling, agent sistemlerinin en küçük yapı taşıdır.
MCP ise bu araçların standartlaştırılmış şekilde sunulmasını sağlar.


#### Araç Çağırma Akışı

Araç çağırma, uygulamanız ile OpenAI API'si aracılığıyla bir model arasında gerçekleşen çok adımlı bir iletişimdir. Araç çağırma akışı beş ana adımdan oluşur:

1. Modelin çağırabileceği araçlarla ilgili bir istekte bulunma
2. Modelden bir araç çağrısı alma
3. Araç çağrısından gelen girdiyle uygulama tarafında kod yürütme
4. Araç çıktısıyla modele ikinci bir istekte bulunma
5. Modelden son bir yanıt alma (veya daha fazla araç çağrısı alma)

![function-calling-diagram-steps](https://cdn.openai.com/API/docs/images/function-calling-diagram-steps.png)



**Referanslar:**
- OpenAI tarafında function calling, modellerin harici sistemler ve verilere bağlanmasını sağlayan mekanizma olarak tanımlanıyor.(https://developers.openai.com/api/docs/guides/function-calling)

### 2. Kurulum

In [ ]:
!pip install requests openai python-dotenv -q

### 3. API Key ayarı

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key giriniz: ")

OpenAI API Key giriniz: ··········


### 4. İlk public API testi: Geocoding

İlk tool’umuz şehir adını koordinata çevirecek.
Çünkü hava durumu API’si şehir ismiyle değil latitude/longitude ile çalışır.

In [ ]:
import requests

def geocode_city(city: str) -> dict:
    url = "https://geocoding-api.open-meteo.com/v1/search"
    params = {
        "name": city,
        "count": 1,
        "language": "tr",
        "format": "json"
    }

    response = requests.get(url, params=params, timeout=10)
    response.raise_for_status()
    data = response.json()

    if "results" not in data:
        return {"error": f"{city} için lokasyon bulunamadı."}

    place = data["results"][0]

    return {
        "name": place["name"],
        "country": place.get("country"),
        "latitude": place["latitude"],
        "longitude": place["longitude"],
        "timezone": place.get("timezone")
    }

geocode_city("İstanbul")

{'name': 'İstanbul',
 'country': 'Türkiye Cumhuriyeti',
 'latitude': 41.01384,
 'longitude': 28.94966,
 'timezone': 'Europe/Istanbul'}

### 5. İkinci public API testi: Weather Forecast

In [ ]:
def get_weather_forecast(latitude: float, longitude: float) -> dict:
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "current": "temperature_2m,relative_humidity_2m,wind_speed_10m,precipitation",
        "daily": "temperature_2m_max,temperature_2m_min,precipitation_probability_max",
        "timezone": "auto",
        "forecast_days": 3
    }

    response = requests.get(url, params=params, timeout=10)
    response.raise_for_status()
    return response.json()


loc = geocode_city("İstanbul")

weather = get_weather_forecast(loc["latitude"], loc["longitude"])

weather["current"]

{'time': '2026-05-30T14:30',
 'interval': 900,
 'temperature_2m': 21.3,
 'relative_humidity_2m': 49,
 'wind_speed_10m': 21.2,
 'precipitation': 0.0}

### 6. Basit karar fonksiyonu

Şimdi LLM olmadan basit bir karar motoru yazıyoruz.
Sonra bu fonksiyonları LLM’e tool olarak vereceğiz.


In [ ]:
def outdoor_activity_decision(weather: dict) -> dict:
    current = weather["current"]
    daily = weather["daily"]

    temp = current["temperature_2m"]
    wind = current["wind_speed_10m"]
    precipitation = current["precipitation"]
    rain_probability = daily["precipitation_probability_max"][0]

    suitable = True
    reasons = []

    if rain_probability >= 60:
        suitable = False
        reasons.append("Yağış ihtimali yüksek.")

    if wind >= 35:
        suitable = False
        reasons.append("Rüzgar hızı yüksek.")

    if temp < 5:
        suitable = False
        reasons.append("Hava çok soğuk.")

    if temp > 35:
        suitable = False
        reasons.append("Hava çok sıcak.")

    if suitable:
        reasons.append("Hava açık hava etkinliği için genel olarak uygun görünüyor.")

    return {
        "suitable": suitable,
        "temperature": temp,
        "wind_speed": wind,
        "precipitation": precipitation,
        "rain_probability": rain_probability,
        "reasons": reasons
    }


outdoor_activity_decision(weather)

{'suitable': True,
 'temperature': 21.3,
 'wind_speed': 21.2,
 'precipitation': 0.0,
 'rain_probability': 0,
 'reasons': ['Hava açık hava etkinliği için genel olarak uygun görünüyor.']}

### 7. Tool Calling’e geçiş

Artık elimizde üç tool var:

1. geocode_city
2. get_weather_forecast
3. outdoor_activity_decision

Şimdi bunları LLM’in çağırabileceği araçlar haline getireceğiz.

**Referanslar:**
OpenAI’nin güncel agent benzeri uygulamalar için önerdiği Responses API; built-in tools, function calling ve remote MCP gibi yetenekleri destekliyor.
(https://developers.openai.com/api/docs/guides/tools)


### 8. OpenAI tool schema

In [ ]:
tools = [
    {
        "type": "function",
        "name": "geocode_city",
        "description": "Verilen şehir adını latitude, longitude ve timezone bilgisine çevirir.",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string",
                    "description": "Şehir adı. Örnek: İstanbul, Ankara, Berlin"
                }
            },
            "required": ["city"],
            "additionalProperties": False
        }
    },
    {
        "type": "function",
        "name": "get_weather_forecast",
        "description": "Latitude ve longitude bilgisine göre güncel hava durumu ve 3 günlük forecast döndürür.",
        "parameters": {
            "type": "object",
            "properties": {
                "latitude": {"type": "number"},
                "longitude": {"type": "number"}
            },
            "required": ["latitude", "longitude"],
            "additionalProperties": False
        }
    }
]

### 9. Tool registry

In [ ]:
available_tools = {
    "geocode_city": geocode_city,
    "get_weather_forecast": get_weather_forecast
}

### 10. İlk cloud LLM tool call

In [ ]:
from openai import OpenAI
import json

client = OpenAI()

user_question = "Yarın İstanbul'da açık hava etkinliği yapmak mantıklı mı?"

response = client.responses.create(
    model="gpt-4.1-mini",
    input=[
        {
            "role": "system",
            "content": "Sen hava durumu verisine göre pratik karar veren bir saha operasyon asistanısın."
        },
        {
            "role": "user",
            "content": user_question
        }
    ],
    tools=tools
)

response.output

[ResponseFunctionToolCall(arguments='{"city":"İstanbul"}', call_id='call_y0HfJTX4mKWwQ7gYc8RBWnOA', name='geocode_city', type='function_call', id='fc_0d5053fd79aa903d006a1acc5f4198819ca926ab33e5437480', namespace=None, status='completed')]

### 11. Tool call yakalama ve çalıştırma

In [ ]:
def run_tool_call(tool_call):
    tool_name = tool_call.name
    arguments = json.loads(tool_call.arguments)

    print(f"Çalıştırılan tool: {tool_name}")
    print(f"Argümanlar: {arguments}")

    result = available_tools[tool_name](**arguments)
    return result

### 12. Basit agent loop

In [ ]:
def run_weather_assistant(user_question: str, max_steps: int = 5):
    input_items = [
        {
            "role": "system",
            "content": """
                Sen gerçek hava durumu verisine göre karar veren bir saha operasyon asistanısın.

                Görevin:
                - Kullanıcının şehir bilgisini anla.
                - Önce geocode_city tool'u ile koordinat al.
                - Sonra get_weather_forecast tool'u ile hava durumunu al.
                - Sonuçlara göre açık hava etkinliği için mantıklı mı değil mi karar ver.
                - Tahmin yürütme, veriye dayan.
                - Cevabını Türkçe ver.
                - Kısa ama gerekçeli cevap ver.
            """
        },
        {
            "role": "user",
            "content": user_question
        }
    ]

    for step in range(max_steps):
        response = client.responses.create(
            model="gpt-4.1-mini",
            input=input_items,
            tools=tools,
        )

        # Model çıktısını conversation history'ye ekle
        input_items.extend(response.output)

        tool_calls = [
            item for item in response.output
            if item.type == "function_call"
        ]

        # Artık tool çağrısı yoksa final cevap hazırdır
        if not tool_calls:
            return response.output_text

        # Modelin istediği tüm tool'ları çalıştır
        for tool_call in tool_calls:
            tool_result = run_tool_call(tool_call)

            input_items.append({
                "type": "function_call_output",
                "call_id": tool_call.call_id,
                "output": json.dumps(tool_result, ensure_ascii=False)
            })

    return "Maksimum agent adımı aşıldı. Muhtemelen model tool çağrısı döngüsünde kaldı."


In [ ]:
answer = run_weather_assistant(
    "Yarın İstanbul'da Tarkan konseri varmış ama Harbiye açık havada. Gidilir mi? Biraz da grip gibiyim."
)

print(answer)

Çalıştırılan tool: geocode_city
Argümanlar: {'city': 'İstanbul'}
Çalıştırılan tool: get_weather_forecast
Argümanlar: {'latitude': 41.01384, 'longitude': 28.94966}
Yarın İstanbul'da hava açık ve yağmur riski çok düşük (%8), sıcaklık da 22.8°C civarında, yani açık hava etkinliği için uygun. Ancak grip belirtileriniz varsa kalabalık ve rüzgarlı dış mekan sizi zorlayabilir. Eğer sağlık durumunuz uygunsa gitmenizde sakınca yok, ama dinlenmenizi de ihmal etmeyin.


### 13. Egzersiz
* “Ankara’da yarın yürüyüş yapılır mı?”
* “İzmir’de açık hava konseri için hava uygun mu?”
* “Berlin’de bisiklet turu yapılır mı?”
* “Rüzgarı özellikle dikkate alarak karar ver.”


### 14. Haftanın mini ödevi
**Ödev:**

Sisteme yeni bir tool ekleyin:

> recommend_activity(decision_result)

Bu tool hava durumuna göre öneri versin:

* yürüyüş
* kapalı mekan
* şemsiye önerisi
* etkinliği ertele
* sabah/akşam önerisi

### 15. Gerçek Hayat Kullanımları


**1. Saha Operasyon Planlama Asistanı**

Bir enerji, telekom veya bakım şirketinde çalışan operasyon yöneticisi olduğunuzu düşünün.

#### Kullanıcı Sorusu

> Yarın Bursa'daki bakım ekibini sahaya göndermeli miyim?

#### Kullanılan Tool'lar

* geocode_city()
* get_weather_forecast()
* field_operation_risk()

#### Beklenen Çıktı

```text
Risk Seviyesi: Düşük

Yağış ihtimali %10.
Rüzgar seviyesi normal.

Planlanan saha operasyonu gerçekleştirilebilir.
```

#### Gerçek Kullanım Alanları

* Telekom şirketleri
* Elektrik dağıtım şirketleri
* Doğalgaz şirketleri
* Belediyeler
* İnşaat firmaları

---

**2. Lojistik Risk Analiz Asistanı**

Lojistik şirketleri hava koşullarını ve rota risklerini değerlendirerek operasyon planlaması yapmaktadır.

#### Kullanıcı Sorusu

> Yarın Ankara'dan Erzurum'a çıkacak sevkiyatlarda hava riski var mı?

#### Kullanılan Tool'lar

* weather_api()
* route_information()
* logistics_risk_analysis()

#### Beklenen Çıktı

```text
Erzurum bölgesinde yoğun yağış bekleniyor.

Tahmini teslimat gecikme riski: %35
```

#### Gerçek Kullanım Alanları

* Trendyol Express
* Hepsijet
* Yurtiçi Kargo
* Aras Kargo
* Sürat Kargo

---

**3. Satış Toplantısı Hazırlık Asistanı**

Satış ekipleri müşteri toplantılarına gitmeden önce şirket hakkında hızlı araştırma yapmak ister.

#### Kullanıcı Sorusu

> Microsoft Türkiye hakkında toplantı öncesi özet hazırla.

#### Kullanılan Tool'lar

* company_search()
* news_search()
* website_scraper()

#### Beklenen Çıktı

```text
Şirket Özeti

Son 30 Günlük Gelişmeler

Potansiyel İhtiyaçlar

Toplantıda Sorulabilecek Sorular
```

#### Gerçek Kullanım Alanları

* Satış ekipleri
* İş geliştirme ekipleri
* Danışmanlık şirketleri

---

**4. Satın Alma Analiz Asistanı**

Kurumsal satın alma ekipleri ürün araştırmalarını hızlandırmak ister.

#### Kullanıcı Sorusu

> 50 adet dizüstü bilgisayar almak istiyoruz. Alternatifleri karşılaştır.

#### Kullanılan Tool'lar

* product_search()
* price_search()
* comparison_tool()

#### Beklenen Çıktı

```text
Toplam Maliyet

Performans Karşılaştırması

Riskler

Önerilen Seçenek
```

#### Gerçek Kullanım Alanları

* Kurumsal IT ekipleri
* Satın alma departmanları

---

**5. Finansal Araştırma Asistanı**

Finans ekipleri şirket raporlarını hızlı analiz etmek ister.

#### Kullanıcı Sorusu

> ASELSAN'ın son çeyrek sonuçlarını özetle.

#### Kullanılan Tool'lar

* financial_report_reader()
* news_search()
* ratio_analysis()

#### Beklenen Çıktı

```text
Gelir Büyümesi

Karlılık

Riskler

Özet Değerlendirme
```

#### Gerçek Kullanım Alanları

* Yatırım şirketleri
* Finans departmanları
* Araştırma ekipleri


---

**6. Siber Güvenlik Analiz Asistanı**

Güvenlik ekipleri büyük miktarda log verisini analiz etmek zorundadır.

#### Kullanıcı Sorusu

> Bu loglarda kritik güvenlik olayı var mı?

#### Kullanılan Tool'lar

* log_reader()
* ioc_lookup()
* risk_classifier()

#### Beklenen Çıktı

```text
Kritiklik: Yüksek

Şüpheli IP Adresi

Olası Saldırı Türü

Önerilen Aksiyonlar
```

#### Gerçek Kullanım Alanları

* SOC ekipleri
* Siber güvenlik firmaları
* Kurumsal IT ekipleri




Bu çalışma dosyası Medeniyet Üniversitesi Teknopark Doğal Dil İşleme ve Büyük Dil Modeleri Dersleri için [Yavuz Kömeçoğlu](https://yavuzkomecoglu.com/) tarafından https://developers.openai.com/api/docs/guides/function-calling ve https://github.com/microsoft/langchain-for-beginners/tree/main/04-function-calling-tools referans alınarak hazırlanmıştır.